# Práctica 01: pronóstico de producción eléctrica

Aplicación de Box–Jenkins al índice mensual de producción eléctrica (`IPG2211A2N`), siguiendo la lógica de clase: graficar, probar estacionariedad, diferenciar, identificar con FAC/FACP, estimar, revisar residuos y pronosticar.

## 1. Librerías

Si se utiliza otro entorno, instalar primero: `%pip install yfinance pandas numpy matplotlib statsmodels`.

In [ ]:
import warnings
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from statsmodels.tools.sm_exceptions import InterpolationWarning

plt.rcParams['figure.figsize'] = (10, 4)
alpha = 0.05

## 2. Llamado al dataset

Se usa como caso principal el mismo dataset mensual de producción eléctrica del material de clase. Al final de esta celda queda comentada la forma de cargar una serie desde Yahoo Finance.

### Frecuencia de los datos

Este dataset eléctrico es mensual, por eso se usa `frecuencia = 'MS'` (`Month Start`). Si trabajas con datos diarios, cambia a `frecuencia = 'D'`. No se debe convertir una serie diaria a mensual salvo que se quiera hacer una agregación; en ese caso se define una regla como `resample('MS').mean()` o `resample('MS').last()`. La frecuencia debe coincidir con el intervalo del dataset.

In [ ]:
# Elige la fuente: 'kaggle', 'yahoo' o 'csv'
fuente = 'kaggle'
frecuencia = 'MS'  # 'MS' mensual; usa 'D' para datos diarios

if fuente == 'kaggle':
    # Dataset Electric_Production.csv usado en clase
    file_path = 'Electric_Production.csv'
    data = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        'shenba/time-series-datasets',
        file_path,
    )
    data['DATE'] = pd.to_datetime(data['DATE'])
    data = data.sort_values('DATE').set_index('DATE')
    serie = data['IPG2211A2N'].astype(float).dropna()
    nombre_serie = 'Producción eléctrica (Kaggle)'

elif fuente == 'csv':
    # CSV local: cambia ruta_csv. Debe tener una fecha y una columna numérica.
    ruta_csv = 'mi_dataset.csv'
    data = pd.read_csv(ruta_csv)
    columna_fecha = 'DATE'       # cambia si tu CSV usa 'Fecha' o 'Date'
    columna_valor = 'IPG2211A2N' # cambia por la variable que deseas analizar
    data[columna_fecha] = pd.to_datetime(data[columna_fecha])
    data = data.sort_values(columna_fecha).set_index(columna_fecha)
    serie = pd.to_numeric(data[columna_valor], errors='coerce').dropna()
    nombre_serie = columna_valor

elif fuente == 'yahoo':
    # Yahoo Finance: ejemplo con una empresa del sector eléctrico
    simbolo = 'NEE'
    data = yf.Ticker(simbolo).history(
        start='2000-01-01', end='2026-01-01', interval='1mo',
        auto_adjust=False, actions=False
    )
    if data.empty:
        raise ValueError('Yahoo no devolvió datos. Revisa el ticker o la conexión.')
    data.index = pd.to_datetime(data.index, utc=True).tz_convert(None)
    data = data.sort_index()
    serie = data['Close'].astype(float).dropna()
    nombre_serie = f'{simbolo} (Close mensual)'
else:
    raise ValueError("fuente debe ser 'kaggle', 'csv' o 'yahoo'")

if frecuencia == 'MS':
    serie.index = pd.DatetimeIndex(serie.index).to_period('M').to_timestamp()
    serie = serie[~serie.index.duplicated(keep='first')].asfreq('MS')
elif frecuencia == 'D':
    serie.index = pd.DatetimeIndex(serie.index).floor('D')
    serie = serie[~serie.index.duplicated(keep='first')].asfreq('D')
else:
    raise ValueError("frecuencia debe ser 'MS' o 'D'")
serie = serie.dropna()
if len(serie) < 80:
    raise ValueError('Se necesitan al menos 80 observaciones.')
print(f'{nombre_serie}: {len(serie)} observaciones ({frecuencia})')
print(f'Desde {serie.index.min().date()} hasta {serie.index.max().date()}')

## 3. Dataset y preparación

In [ ]:
print('Dimensiones:', data.shape)
print('Periodo:', data.index.min().date(), 'a', data.index.max().date())
display(data.head())
display(data.tail())
data.info()

In [ ]:
display(data.describe().T)
display(data.isna().sum().rename('Nulos').to_frame())
print('Fechas duplicadas:', data.index.duplicated().sum())

In [ ]:
data = data.loc[~data.index.duplicated(keep='first')].copy()
serie = serie.astype(float).dropna().asfreq(frecuencia)
if (serie <= 0).any(): raise ValueError('Hay valores no positivos.')
serie.plot(title='Producción eléctrica mensual', ylabel='Índice', xlabel='Fecha')
plt.tight_layout(); plt.show()

## 4. Fase 1: identificación y estacionariedad

### 4.1. División temporal

In [ ]:
corte = int(len(serie) * 0.8)
train = serie.iloc[:corte]
test = serie.iloc[corte:]

display(pd.DataFrame({
    'Datos': [len(train), len(test)],
    'Desde': [train.index.min().date(), test.index.min().date()],
    'Hasta': [train.index.max().date(), test.index.max().date()]
}, index=['Entrenamiento', 'Prueba']))

train.plot(label='Entrenamiento')
test.plot(label='Prueba')
plt.title('División de la serie')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Estacionariedad

ADF: H₀ = la serie no es estacionaria; Hₐ = la serie es estacionaria. Si p-valor < 0,05 se rechaza H₀.

In [ ]:
def prueba_adf(x, nombre):
    r = adfuller(x.dropna(), autolag='AIC', result_object=False)
    print(f'{nombre}: ADF={r[0]:.4f}; p-valor={r[1]:.4f};', 'estacionaria' if r[1] < alpha else 'no estacionaria')
    return r[1]
corte = int(len(serie) * 0.8)
train, test = serie.iloc[:corte], serie.iloc[corte:]
p0 = prueba_adf(train, 'Original')

### Diferenciación

Se revisan d=0, d=1 y d=2. Se usa el menor orden que pase ADF.

In [ ]:
train_diff = train.diff().dropna()
train_diff2 = train_diff.diff().dropna()
series = [train, train_diff, train_diff2]
nombres = ['Original (d=0)', 'Primera diferencia (d=1)', 'Segunda diferencia (d=2)']

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for x, nombre, ax in zip(series, nombres, axes):
    x.plot(ax=ax, title=nombre)
    ax.set_ylabel('USD')
plt.tight_layout()
plt.show()

In [ ]:
p1 = prueba_adf(train_diff, 'Primera diferencia')
p2 = prueba_adf(train_diff2, 'Segunda diferencia')
if p0 < alpha: d = 0
elif p1 < alpha: d = 1
elif p2 < alpha: d = 2
else: raise ValueError('No se logró estacionariedad con d=0, 1 o 2.')
serie_estacionaria = [train, train_diff, train_diff2][d]
print(f'Orden elegido: d={d}')
serie_estacionaria = [train, train_diff, train_diff2][d]


### 4.6. FAC y FACP para identificar p y q

| Modelo sobre la serie estacionaria | FAC | FACP |
|---|---|---|
| AR(p) | Decae gradualmente | Corte después de p |
| MA(q) | Corte después de q | Decae gradualmente |
| ARMA(p,q) | Decae gradualmente | Decae gradualmente |

Estos patrones son orientativos. Se comparan órdenes pequeños con AIC y BIC para apoyar la elección.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
plot_acf(serie_estacionaria, lags=30, zero=False, ax=axes[0], title=f'FAC: d={d}')
plot_pacf(serie_estacionaria, lags=30, zero=False, method='ywm', ax=axes[1],
          title=f'FACP: d={d}')
plt.tight_layout()
plt.show()

## 5. Fase 2: estimación y comparación de modelos

Con `d=0`, ARIMA(p,0,0) es AR, ARIMA(0,0,q) es MA y ARIMA(p,0,q) es ARMA. Con `d>0`, el modelo de la serie original es **ARIMA(p,d,q)**, aunque las diferencias se modelen con AR, MA o ARMA.

Se prueban `p` y `q` entre 0 y 3. Se incluye ARIMA(0,1,0), un paseo aleatorio sin deriva, cuando `d=1`. Se ajusta **train original**: ARIMA realiza las diferencias internamente y devuelve pronósticos en USD. Se usa constante si `d=0` y sin tendencia determinista si `d>0`.

In [ ]:
modelos, resultados = {}, []
trend = 'c' if d == 0 else 'n'
for p in range(3):
    for q in range(3):
        orden = (p, d, q)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                ajuste = ARIMA(train, order=orden, trend=trend, freq=frecuencia).fit(method_kwargs={'maxiter': 100})
            if np.isfinite(ajuste.aic):
                modelos[orden] = ajuste
                resultados.append({'Orden': orden, 'AIC': ajuste.aic, 'BIC': ajuste.bic})
        except (ValueError, np.linalg.LinAlgError): pass
if not resultados: raise ValueError('No se pudo ajustar ningún modelo.')
comparacion = pd.DataFrame(resultados).sort_values('AIC').reset_index(drop=True)
display(comparacion)
orden_elegido = tuple(comparacion.loc[0, 'Orden'])
modelo = modelos[orden_elegido]
print('Modelo elegido por AIC: ARIMA', orden_elegido)
nombre = f'ARIMA{orden_elegido}'


## 6. Fase 3: diagnóstico de residuos

### 6.1. Selección con AIC y Ljung–Box

Un AIC menor indica un mejor equilibrio entre ajuste y complejidad dentro de esta comparación. Siguiendo Box–Jenkins, se revisan los modelos en ese orden y se elige el primero sin autocorrelación residual detectada a 10 y 20 rezagos.

Ljung–Box: H₀ = no hay autocorrelación hasta el rezago evaluado. Se descartan los residuos iniciales de inicialización y se descuentan `p+q` grados de libertad. Pasar esta prueba no garantiza independencia ni varianza constante.

In [ ]:
residuales = modelo.resid.dropna()
print('Residuos disponibles:', len(residuales))

# Gráficos de los errores del modelo
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
residuales.plot(ax=axes[0, 0], title='Residuos en el tiempo')
axes[0, 0].axhline(0, color='black', linestyle='--')
plot_acf(residuales, lags=24, zero=False, ax=axes[0, 1], title='FAC de residuos')
plot_pacf(residuales, lags=24, zero=False, method='ywm', ax=axes[1, 0], title='FACP de residuos')
axes[1, 1].hist(residuales, bins=20, edgecolor='black')
axes[1, 1].set_title('Histograma de residuos')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
qqplot(residuales, line='s', ax=ax)
ax.set_title('Q–Q de residuos')
plt.tight_layout(); plt.show()


### 6.2. Gráficos de los residuos

### 6.3. Ljung–Box, ARCH y Jarque–Bera

ARCH contrasta ausencia de heterocedasticidad condicional; Jarque–Bera contrasta normalidad. Si p < 0,05 se rechaza su hipótesis nula. Una media próxima a cero se revisa descriptivamente.

In [ ]:
lb = acorr_ljungbox(residuales, lags=[10], return_df=True)
arch = het_arch(residuales, nlags=10, result_object=False)
jb = jarque_bera(residuales)
pruebas = pd.DataFrame({'Prueba':['Ljung–Box','ARCH','Jarque–Bera'], 'Estadístico':[lb['lb_stat'].iloc[0],arch[0],jb[0]], 'p-valor':[lb['lb_pvalue'].iloc[0],arch[1],jb[1]]})
display(pruebas)
print(f'Media de residuos: {residuales.mean():.4f}')

## 7. Fase 4: pronóstico y evaluación

### 7.1. Pronóstico del conjunto de prueba

El modelo queda fijado antes de consultar los errores de prueba. Se pronostica todo el bloque desde el último día de entrenamiento, sin actualizar con valores de `test`. Se compara con una referencia ingenua: repetir el último cierre conocido.

Los intervalos del 95 % dependen de los supuestos del modelo; su cobertura puede ser inadecuada si ARCH o Jarque–Bera rechazan sus hipótesis.

In [ ]:
pronostico = modelo.get_forecast(steps=len(test))
intervalo = pronostico.conf_int(alpha=alpha)
prediccion = pronostico.predicted_mean

evaluacion = pd.DataFrame({'Real': test, 'Pronóstico': prediccion,
                           'Ingenuo': train.iloc[-1],
                           'Inferior 95%': intervalo.iloc[:, 0],
                           'Superior 95%': intervalo.iloc[:, 1]})
display(evaluacion.head())
display(evaluacion.tail())

plt.plot(pd.concat([train.tail(90), test]), label='Real')
plt.plot(prediccion, label=nombre, linestyle='--')
plt.plot(test.index, evaluacion['Ingenuo'], label='Ingenuo', linestyle=':')
plt.fill_between(test.index, intervalo.iloc[:, 0], intervalo.iloc[:, 1],
                 alpha=0.2, label='Intervalo del 95 %')
plt.title('Pronóstico sobre el conjunto de prueba')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.show()

### 7.2. Errores del pronóstico

MSE, RMSE, MAE y MAPE: cuanto menores, mejor. RMSE y MAE se expresan en USD, MSE en USD² y MAPE en porcentaje. Las métricas del bloque completo se complementan con sus primeros 30 días, un horizonte más corto.

In [ ]:
def metricas(real, estimado):
    error = real - estimado
    mse = np.mean(error ** 2)
    return {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': np.mean(np.abs(error)),
            'MAPE (%)': np.mean(np.abs(error / real)) * 100}

metricas_test = pd.DataFrame({
    nombre: metricas(test, evaluacion['Pronóstico']),
    'Ingenuo': metricas(test, evaluacion['Ingenuo'])
}).T
print('Todo el conjunto de prueba:')
display(metricas_test.round(2))
print('Primeros 30 días de prueba:')
display(pd.DataFrame({
    nombre: metricas(test.iloc[:30], evaluacion['Pronóstico'].iloc[:30]),
    'Ingenuo': metricas(test.iloc[:30], evaluacion['Ingenuo'].iloc[:30])
}).T.round(2))

# Error de pronóstico: valor observado menos valor pronosticado
error_pronostico = test - evaluacion['Pronóstico']
error_pronostico.plot(title='Error del pronóstico en el conjunto de prueba', ylabel='Error')
plt.axhline(0, color='black', linestyle='--')
plt.tight_layout(); plt.show()


### 7.3. Reajuste y pronóstico de 30 días posteriores al dataset

Se conserva el orden elegido y se reestiman los parámetros con toda la serie. El horizonte empieza después de la última fecha descargada; este pronóstico no es una evaluación con datos observados.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    modelo_final = ARIMA(serie, order=orden_elegido, trend=trend, freq=frecuencia).fit(method_kwargs={'maxiter': 100})
if not modelo_final.mle_retvals.get('converged', False):
    raise ValueError('El reajuste final no convergió. Revisar antes de pronosticar.')

futuro = modelo_final.get_forecast(steps=30)
intervalo_futuro = futuro.conf_int(alpha=alpha)
tabla_futuro = pd.DataFrame({'Pronóstico': futuro.predicted_mean,
                             'Inferior 95%': intervalo_futuro.iloc[:, 0],
                             'Superior 95%': intervalo_futuro.iloc[:, 1]})
display(tabla_futuro.round(2))

plt.plot(serie.tail(90), label='Real')
plt.plot(tabla_futuro['Pronóstico'], label='Pronóstico', linestyle='--')
plt.fill_between(tabla_futuro.index, tabla_futuro['Inferior 95%'],
                 tabla_futuro['Superior 95%'], alpha=0.2, label='Intervalo del 95 %')
plt.title(f'ARIMA{orden_elegido}: 30 días después del dataset')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Conclusiones

In [ ]:
print(f'1. Se analizaron {len(serie)} observaciones mensuales de producción eléctrica.')
print(f'2. Se seleccionó d={d}, según la prueba ADF.')
print(f'3. El modelo elegido por AIC fue ARIMA{orden_elegido}.')
print(f'4. RMSE del pronóstico: {metricas_test.loc[nombre, "RMSE"]:.2f}.')
print('5. Las pruebas de residuos se muestran en la tabla anterior.')
print('6. El pronóstico es una estimación; no reproduce exactamente cada variación futura.')